# 02 — Logistic Regression From First Principles

This notebook implements Logistic Regression from scratch using NumPy: sigmoid, binary cross-entropy, gradient descent, L2 regularization, thresholding, and classification metrics.

In [ ]:
import numpy as np

## 1. Create Binary Classification Data

In [ ]:
rng = np.random.default_rng(42)

n = 260
class0 = rng.multivariate_normal([-1.5, -1.0], [[0.9, 0.25], [0.25, 0.8]], size=n // 2)
class1 = rng.multivariate_normal([1.4, 1.2], [[0.9, -0.2], [-0.2, 0.9]], size=n // 2)

X = np.vstack([class0, class1])
y = np.array([0] * (n // 2) + [1] * (n // 2))

X.shape, y.shape

## 2. Split and Standardize

In [ ]:
def train_test_split_numpy(X, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    test_n = int(n * test_size)
    test_idx = indices[:test_n]
    train_idx = indices[test_n:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std == 0, 1, std)
    return (X_train - mean) / std, (X_test - mean) / std

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y)
X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

X_train_scaled.shape, X_test_scaled.shape

## 3. Sigmoid

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

In [ ]:
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1 / (1 + np.exp(-z))

sigmoid(np.array([-3, 0, 3]))

## 4. Binary Cross-Entropy

$$
\mathcal{L}= -\frac{1}{n}\sum_i [y_i\log p_i+(1-y_i)\log(1-p_i)]
$$

In [ ]:
def binary_cross_entropy(y_true, p_pred, eps=1e-15):
    p_pred = np.clip(p_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(p_pred) + (1 - y_true) * np.log(1 - p_pred))

## 5. Train Logistic Regression

Gradient:

$$
\nabla_\beta \mathcal{L}=\frac{1}{n}X^T(p-y)
$$

In [ ]:
def add_bias_column(X):
    return np.column_stack([np.ones(X.shape[0]), X])


def train_logistic_regression_gd(X, y, lr=0.3, steps=3000, lambda_=0.01):
    X_bias = add_bias_column(X)
    beta = np.zeros(X_bias.shape[1])
    losses = []

    for _ in range(steps):
        logits = X_bias @ beta
        probabilities = sigmoid(logits)
        loss = binary_cross_entropy(y, probabilities) + lambda_ * np.sum(beta[1:] ** 2)
        losses.append(loss)
        gradient = (1 / len(y)) * X_bias.T @ (probabilities - y)
        regularization = np.zeros_like(beta)
        regularization[1:] = 2 * lambda_ * beta[1:]
        beta = beta - lr * (gradient + regularization)

    return beta, np.array(losses)

beta, losses = train_logistic_regression_gd(X_train_scaled, y_train)

beta, losses[0], losses[-1]

## 6. Predict Probabilities and Classes

In [ ]:
def predict_proba(X, beta):
    X_bias = add_bias_column(X)
    return sigmoid(X_bias @ beta)


def predict_class(X, beta, threshold=0.5):
    return (predict_proba(X, beta) >= threshold).astype(int)

probabilities = predict_proba(X_test_scaled, beta)
predictions = predict_class(X_test_scaled, beta, threshold=0.5)

probabilities[:5], predictions[:5]

## 7. Classification Metrics

In [ ]:
def classification_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return accuracy, precision, recall, f1, (tp, tn, fp, fn)

classification_metrics(y_test, predictions)

## 8. Threshold Comparison

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    pred = (probabilities >= threshold).astype(int)
    print(threshold, classification_metrics(y_test, pred))

## Reflection

Logistic Regression learns probabilities. The threshold turns those probabilities into decisions.